# Lecture 15

## Pandas: Summaries with Pivot Tables and Group by

## Week 5 Friday

## Miles Chen, PhD

Adapted from Python Data Science Handbook by Jake VanderPlas and Python for Data Analysis by Wes McKinney

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
pd.__version__

# Some important data transformation tools

## Multi Index, Hierarchical Indexing

In [ ]:
# manual multi-index creation:
my_index = pd.MultiIndex.from_arrays(
    [
    ['a','a','a','b','b','b','c','c','c'],
    [ 4 , 5 , 6 , 4 , 5 , 6 , 4 , 5 , 6 ]
])

data = pd.Series([4, 5, 6, 8, 10, 12, 12, 15, 18], index=my_index)

In [ ]:
data

In [ ]:
data.index

In [ ]:
# select via the outer index
data.loc['b']

In [ ]:
# select via the inner index
data.loc[:,5] 

In [ ]:
type(data.loc[:,5])

In [ ]:
data.loc[:,5].index

In [ ]:
# the unstack function returns a new DataFrame where the values have been unstacked
# similar to tidyr's spread()/pivot_wider function in R
data.unstack()

In [ ]:
# after unstacking, the index is no longer a multi index
data.unstack().index

In [ ]:
data.unstack().shape

In [ ]:
# the inverse operation of unstack() is stack()
# applying both of these functions will return the same series
data.unstack().stack()

In [ ]:
# you can swap the levels of the multi index using swaplevel
data.swaplevel()

In [ ]:
# the .loc accessors work as expected
data.swaplevel().loc[:,'a']

In [ ]:
# swaplevel will keep the original order
# you may want to sort based on the new swapped index levels
# you must save the output as data remains unchanged
data.swaplevel().sort_index()

In [ ]:
print(data)

In [ ]:
data.swaplevel().unstack()

In [ ]:
# compare to:
data.unstack()

In [ ]:
# summing and other aggregate functions can be performed on an index-based level
# calling sum() on a series, will sum the whole series
data.sum()

In [ ]:
# you can call groupby on level 0 (the first level of the index) and then sum
# we get sums for each value in the first level of the index
# we will cover groupby in more detail later
data.groupby(level = 0).sum()

In [ ]:
data.groupby(level = 1).sum()

# Reshaping and Pivoting Data

In [ ]:
data = pd.DataFrame(np.arange(1,7).reshape((2, 3)),
                    index  = pd.Index(['alpha', 'beta'], name='letter'),
                    columns= pd.Index(['one', 'two', 'three'], name = 'number'))
data

In [ ]:
data.stack()  # creates a multi-index

In [ ]:
data.stack().unstack()  # unstack undoes the creation of the stacks

In [ ]:
data.stack().unstack(0) # you can specify how the unstacking should be done
# here we specify that we should unstack the first level of the multi-index

In [ ]:
data.stack().unstack('letter')
# you can specify the unstacking by the index level name

In [ ]:
data.stack().unstack('number')

### Unstacking can introduce missing values

In [ ]:
s1 = pd.Series([0, 1, 2, 3], index=['a', 'b', 'c', 'd'])
s2 = pd.Series([4, 5, 6], index=['c', 'd', 'e'])
data2 = pd.concat([s1, s2], keys=['one', 'two'])  
# using the argument keys when concat series will produce a multi-index
data2

In [ ]:
data2.unstack()

In [ ]:
data2.unstack().stack() # stack() will filter out missing values

# Small example data wrangling

In [ ]:
data = pd.read_csv('../data/macrodata.csv')

https://www.statsmodels.org/dev/datasets/generated/macrodata.html

In [ ]:
data.info()

In [ ]:
data.head()

https://pandas.pydata.org/pandas-docs/stable/generated/pandas.PeriodIndex.html

In [ ]:
# We can create a time based index of periods consisting of the year and quarter
periods = pd.PeriodIndex.from_fields(year = data.year, quarter = data.quarter)

In [ ]:
periods

In [ ]:
columns = pd.Index(['realgdp', 'infl', 'unemp'], name = 'item')
columns

In [ ]:
data = data.reindex(columns = columns) # forces columns to conform to the column index we specified

In [ ]:
data.head(10)

In [ ]:
periods.to_timestamp('D','start')  # changes 1959Q1 to a date: the start date of Q1 of 1959: 1959-01-01

In [ ]:
# the current index is just integers, and we want to replace it
data.index

In [ ]:
# specify a new index directly
data.index = periods.to_timestamp('D','start')

In [ ]:
data.head()

In [ ]:
data.reset_index() # resets the index to the default integer index and moves the current index into a 
# column named 'index'

In [ ]:
# pandas has a melt function that is similar to tidyr's pivot_longer() function in R
# It takes values in multiple columns and stacks them into a single column, 
# while keeping the other columns as identifier variables

ldata = data.reset_index().melt(
    id_vars='index', 
    var_name='item', 
    value_name='value'
)

In [ ]:

ldata = ldata.rename(columns={'index': 'date'})

In [ ]:
ldata

In [ ]:
# the shape of ldata is 609 by 3
# there are 609 rows - each date has three rows
# the three columns are date, item name, value
ldata.shape

In [ ]:
# unstack doesn't work, because the stacking and unstacking is powered by multi-index
# instead, all the dates, item names, and values get 'flowed' into one column
# Notice the length is now 1827
ldata.unstack()

https://pandas.pydata.org/pandas-docs/stable/generated/pandas.DataFrame.pivot.html

#### you must specify index, columns and values in pivot

In [ ]:
# if the data is in 'long' form, you can change it to 'wide' form with pivot
# the argument columns specifies the column to use to make the new columns, 
# and values specifies the column to use for populating the new values
# .pivot() is the inverse of melt()
# You can think of .pivot() as being similar to tidyverse's pivot_wider() function
ldata.pivot(index = 'date',columns = 'item',values = 'value').head()

In [ ]:
# if the data is in 'long' form, you can change it to 'wide' form with pivot
# in this example, we specify the index to be 'item', of which there are three
# the date values become the columns, of which there are 203
ldata.pivot(index = 'item',columns = 'date',values = 'value').head()

In [ ]:
# all of these pivot operations return new objects and leave the original data unchanged
data.head()

# Pivot_table()

Pandas has another function called `pivot_table()`

`pivot()` is strictly for rearranging data, while `pivot_table()` is for rearranging **and** aggregating data.

`pivot_table()` will always take data in long form and make it wide. The argument `columns` specifies where the column names will come from.

In [ ]:
# long-format data
df = pd.DataFrame({
    'Date': ['Jan 1', 'Jan 1', 'Jan 1', 'Jan 2'],
    'Fruit': ['Apple', 'Apple', 'Banana', 'Apple'],
    'Sold': [5, 3, 8, 6]
})

print(df)

If we try to reshape this using `pivot()`, Pandas will hit the two Apple entries for Jan 1 and panic. 

It asks: "Do I put the 5 or the 3 in the intersection of 'Jan 1' and 'Apple'?" Because it doesn't know how to combine them, it throws an error.

In [ ]:
# This will crash!
df.pivot(index='Date', columns='Fruit', values='Sold')

# Error: ValueError: Index contains duplicate entries, cannot reshape

`pivot_table()` is built to handle this situation. When it finds duplicate index/column pairs, it applies an aggregation function to combine them.

By default, it calculates the mean, but for sales data, we usually want to add them together using `aggfunc='sum'`.

In [ ]:
# pivot_table combines the 5 and the 3
wide_df = df.pivot_table(
    index='Date', 
    columns='Fruit', 
    values='Sold', 
    aggfunc='sum',     # Tell Pandas HOW to combine duplicates
    fill_value=0       # Clean up any missing (NaN) values with 0
)

print(wide_df)

# Group By


In [ ]:
pd.__version__

In [ ]:
np.random.seed(1)
df = pd.DataFrame({'key1' : ['a', 'a', 'b', 'b', 'a'],
                   'key2' : ['one', 'two', 'one', 'two', 'one'],
                   'data1' : np.random.randint(20, size = 5),
                   'data2' : np.random.randint(20, size = 5)})
df

In [ ]:
grouped = df['data1'].groupby(df['key1'])
grouped

In [ ]:
grouped.mean()

In [ ]:
df

### if there is a mix of numeric and categorical data, specify `numeric_only = True`

In [ ]:
df.groupby(by = 'key1').mean(numeric_only = True)
# if you don't specify the column, it'll apply the function to the entire dataframe

In [ ]:
df

In [ ]:
means = df['data1'].groupby([df['key1'], df['key2']]).mean()
means
# means has a multi-index

In [ ]:
# with the multi-index, you can unstack
means.unstack()

In [ ]:
df

In [ ]:
# you can perform group by on Series that are not in the dataframe, but are of the correct length
states = np.array(['Ohio', 'California', 'California', 'Ohio', 'Ohio'])
years = np.array([2005, 2005, 2006, 2005, 2006])
df['data1'].groupby([states, years]).mean()

In [ ]:
df

In [ ]:
df.groupby(['key1', 'key2']).size() # you don't always have to use mean, you can use other functions as well

### Iterating over groups

In [ ]:
df

In [ ]:
# the groupby creates a series of tuples that can be unpacked into name and group
for name, group in df.groupby('key1'):
    print("name:", name)
    print('------')
    print("group:\n", group)
    print('------')
    print("data1 mean:", group.data1.mean())
    print("data2 mean:", group.data2.mean())
    print('**************************')

In [ ]:
for name, group in df.groupby('key2'):
    print("name:", name)
    print('------')
    print("group:\n", group)
    print('------')
    print("data1 mean:", group.data1.mean())
    print("data2 mean:", group.data2.mean())
    print('**************************')
